# Data Cleaning and Executive EDA
[Open in Colab](https://colab.research.google.com/github/ericmavigo/retail-demand-forecasting/blob/main/notebooks/02_data_cleaning_and_eda.ipynb)

This notebook builds trusted analytical tables and explains five years of store, category, product, event and seasonal performance.

In [ ]:
# Run once in a fresh environment
# %pip install -r ../requirements-dev.txt

In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
import plotly.express as px
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
RAW=ROOT/'data'/'raw'; PROCESSED=ROOT/'data'/'processed'

## 1. Download and validate
Accept the M5 rules and configure `KAGGLE_API_TOKEN`; raw data remains outside Git.

In [ ]:
subprocess.run([sys.executable,str(ROOT/'src/download_data.py'),'--output-dir',str(RAW)],check=True)
subprocess.run([sys.executable,str(ROOT/'src/data_audit.py'),'--data-dir',str(RAW),'--output-dir',str(ROOT/'reports')],check=True)

## 2. Clean and join sales, prices and calendar
Zeros are retained. The pipeline distinguishes real zero demand from quality problems and validates price coverage before estimating revenue.

In [ ]:
subprocess.run([sys.executable,str(ROOT/'src/build_analytics.py'),'--data-dir',str(RAW),'--output-dir',str(PROCESSED)],check=True)
json.loads((PROCESSED/'validation.json').read_text())

In [ ]:
daily=pd.read_csv(PROCESSED/'daily_overview.csv',parse_dates=['date'])
weekly=pd.read_csv(PROCESSED/'weekly_overview.csv',parse_dates=['date'])
stores=pd.read_csv(PROCESSED/'store_summary.csv')
products=pd.read_csv(PROCESSED/'product_summary.csv')
categories=pd.read_csv(PROCESSED/'category_summary.csv')

## 3. Executive overview

In [ ]:
display(pd.Series({'Units sold':daily.units.sum(),'Estimated revenue':daily.estimated_revenue.sum(),'Stores':len(stores),'Products':len(products)}).to_frame('value'))
px.bar(stores.sort_values('estimated_revenue'),x='estimated_revenue',y='store_id',orientation='h',title='Estimated revenue by store').show()

## 4. Five-year seasonality

In [ ]:
px.line(daily,x='date',y=['units','moving_average_28'],title='Daily demand and 28-day trend').show()
px.line(weekly[weekly.week_of_year<=52],x='week_of_year',y='units',color='year',title='Yearly demand curves').show()

## 5. Product portfolio and Pareto

In [ ]:
px.bar(categories,x='cat_id',y='estimated_revenue',color='cat_id',title='Revenue by category').show()
p=products.copy(); p['product_share']=(p.index+1)/len(p)
px.line(p,x='product_share',y='cumulative_revenue_share',title='Product revenue Pareto curve').show()
products.head(20)